In [1]:
!pip install -q transformers torch

In [2]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

model_name = "distilbert-base-cased-distilled-squad"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

print("Question Answering model loaded successfully!")

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Question Answering model loaded successfully!


In [3]:
def answer_question(context, question):

    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    start_scores = outputs.start_logits
    end_scores = outputs.end_logits

    start_index = torch.argmax(start_scores)
    end_index = torch.argmax(end_scores)

    if end_index < start_index:
        end_index = start_index

    answer_tokens = inputs["input_ids"][0][start_index:end_index + 1]

    answer = tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    )

    confidence = (
        torch.softmax(start_scores, dim=1)[0][start_index] *
        torch.softmax(end_scores, dim=1)[0][end_index]
    )

    return answer, confidence.item()

In [4]:
context = """
Artificial intelligence is a field of computer science that focuses
on creating systems capable of performing tasks that normally require
human intelligence. Machine learning is a subset of artificial
intelligence that allows computers to learn patterns from data.
Deep learning is a further subset of machine learning that uses
neural networks with multiple layers.
"""

question = "What is machine learning?"

answer, confidence = answer_question(context, question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

print(f"\nConfidence: {confidence:.4f}")

Question:
What is machine learning?

Answer:
a subset of artificial intelligence

Confidence: 0.2885


In [6]:
print("=" * 60)
print("🧠 AI QUESTION ANSWERING SYSTEM")
print("=" * 60)

context = input("\nEnter the information/context:\n")

while True:

    question = input("\nAsk a question (or type 'exit'): ")

    if question.strip().lower() == "exit":
        print("\nQuestion answering system closed.")
        break

    if not question.strip():
        print("Please enter a question.")
        continue

    answer, confidence = answer_question(context, question)

    print("\n" + "-" * 60)
    print("ANSWER")
    print("-" * 60)
    print(answer)

    print(f"\nConfidence: {confidence:.2%}")

🧠 AI QUESTION ANSWERING SYSTEM

Enter the information/context:
Google was founded in 1998 by Larry Page and Sergey Brin while they were students at Stanford University. The company initially focused on developing a search engine. Google later expanded into areas such as online advertising, cloud computing, mobile operating systems, artificial intelligence, and hardware. Its headquarters are located in Mountain View, California.

Ask a question (or type 'exit'): who found google?

------------------------------------------------------------
ANSWER
------------------------------------------------------------
Larry Page and Sergey Brin

Confidence: 99.11%

Ask a question (or type 'exit'): exit

Question answering system closed.
